# Lumis 16: Can a skin foundation model read phone photos better?

Model v1 (EfficientNet-B0 trained on DermNet) is right about 38% of the time on SCIN phone photos. Fine-tuning it on
SCIN (v2, v2b) did not pass its rules. This notebook tests a different approach: start from models that were already
trained on large amounts of skin images, including smartphone photos, and train only a small classifier on top.

**Candidates** (each is a logistic regression on the model's image embedding, trained on DermNet training photos plus
SCIN phone photos):
- **Derm Foundation** (Google, 6144-number embedding, 448 x 448 input)
- **MedSigLIP** (Google, medical image encoder, 448 x 448 input)
- each **with** and **without** extra SCIN cases that have weaker dermatologist agreement (main label one of the five
  conditions with agreement 0.25 to 0.5; weighted by their agreement; used for training only, never for scoring).

**Comparison** (also reported, never deployed by this notebook): **Claude Haiku 5.5 asked directly** which of the five
conditions a photo shows (no Lumis model), to test whether Lumis does better than a general AI.

**Rules fixed 2026-11-12, before this notebook ran:**
1. **Scoring:** 5-fold cross-validation over all SCIN train + validation cases (482 cases), split by case and grouped
   so identical photos stay together, stratified by condition. One close-up photo per case is scored. v1 is scored on
   the same cases. The SCIN test split is not loaded here; it is used once, later, for the final model only.
2. **A candidate replaces v1 only if**, on those 482 cases (v1's prior adjustment applied to every model, as before):
   average sensitivity across the five conditions is higher, macro F1 is higher, tinea sensitivity is not lower,
   accuracy is not lower, and DermNet validation accuracy drops by at most 3 points.
3. **If several pass**, the one with the highest average sensitivity wins (ties: the one without extra cases).
   If none pass, v1 stays.
4. The classifier's strength (C) is picked inside each training fold (3-fold, by average sensitivity), never on the
   scored cases. Embedding size is reduced to 256 with PCA fitted on each training fold.

Disclosed: the 121 SCIN validation cases were already looked at in notebooks 12 to 15; Google does not list every
image its models were trained on, so some SCIN or DermNet photos may have been seen by them.

**Runtime: GPU (T4).** Needs Colab secrets `HF_TOKEN` (terms accepted for google/derm-foundation and
google/medsiglip-448) and `ANTHROPIC_API_KEY`. MedSigLIP runs on the GPU in minutes. **Derm Foundation can only run on
the CPU** (Google exported it that way), so it is slow: the notebook prints the time left after each chunk of 256
photos. Every chunk is saved to Drive, so if Colab disconnects, run all again and it continues where it stopped.
The Claude comparison costs about $0.25.

In [ ]:
SEED = 42
N_FOLDS = 5
PCA_DIM = 256
C_GRID = [0.01, 0.1, 1.0]
SCIN_W = 3.0                 # SCIN photos count 3x relative to DermNet (same as v2b's SCIN_REPEAT)
MAX_WEIGHT = 5.0             # cap on class-balance weight within SCIN (scabies is rare)
MIN_EXTRA_AGREEMENT = 0.25
BACKBONES = ['derm_foundation', 'medsiglip']
ZERO_SHOT_MODELS = ['claude-haiku-5-5']     # add 'claude-sonnet-5-5' to also compare a larger model (about $2 more)
MAX_DERMNET_DROP = 0.03

In [ ]:
!pip install -q google-cloud-storage anthropic
from google.colab import drive, userdata
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'
HF_TOKEN = userdata.get('HF_TOKEN')
ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')

In [ ]:
import os, io, json, time, zipfile, hashlib, glob
import numpy as np, pandas as pd

OUT = f'{BASE}/exports/foundation'
os.makedirs(OUT, exist_ok=True)
if os.path.exists(f'{OUT}/decision.json'):
    raise RuntimeError(f'{OUT}/decision.json exists: the comparison was already decided. Do not rerun it to try again.')

for z, marker in [(f'{BASE}/data/dermnet.zip', '/content/dermnet'), (f'{BASE}/data/scin_images.zip', '/content/scin')]:
    if not os.path.exists(marker):
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/dermnet' if 'dermnet' in z else '/content')

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
V1_RUN = runs[(runs.backbone == 'efficientnetb0') & (runs.split == 'v2')].iloc[-1]['run']
v1info = runs[runs.run == V1_RUN].iloc[-1]
cal = json.load(open(f'{BASE}/exports/runs/{V1_RUN}/calibration.json'))
CLASSES = cal['classes']
K = len(CLASSES)
IDX = {c: i for i, c in enumerate(CLASSES)}
ADJ = np.array(cal['logit_adjust'])
TINEA = IDX['tinea']

derm = pd.read_csv(f'{BASE}/data/lumis_split_v2.csv')
d_train = derm[derm.split == 'train'].reset_index(drop=True)
d_val = derm[derm.split == 'val'].reset_index(drop=True)
d_train['file'] = '/content/dermnet/' + d_train.path
d_val['file'] = '/content/dermnet/' + d_val.path
split = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
tv = split[split.split.isin(['train', 'val'])].reset_index(drop=True)          # SCIN test is never used here
tv['file'] = '/content/' + tv.path
print(f'v1 = {V1_RUN} | DermNet train {len(d_train)}, val {len(d_val)} | SCIN train+val {len(tv)} photos, '
      f'{tv.case_id.nunique()} cases')

## 1. Extra SCIN cases with weaker agreement (training only, saved once)

In [ ]:
import os, io, ast, json, time, hashlib, zipfile
import numpy as np, pandas as pd
from google.cloud import storage

BUCKET = 'dx-scin-public-data'
try:
    bucket = storage.Client.create_anonymous_client().bucket(BUCKET)
    bucket.blob('dataset/scin_cases.csv').download_as_bytes(start=0, end=10)
except Exception as e:
    print('Anonymous access failed, signing in instead (a popup will ask you to allow access):', type(e).__name__)
    from google.colab import auth
    auth.authenticate_user()
    bucket = storage.Client(project='dx-scin-public').bucket(BUCKET)

def fetch(path):
    return bucket.blob(path).download_as_bytes()

cases = pd.read_csv(io.BytesIO(fetch('dataset/scin_cases.csv')), dtype={'case_id': str})
labels = pd.read_csv(io.BytesIO(fetch('dataset/scin_labels.csv')), dtype={'case_id': str})
scin = cases.merge(labels, on='case_id')
print(len(scin), 'SCIN cases')

In [ ]:
def parse(x):
    if isinstance(x, str) and x.strip():
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return {}
    return {}

def to_lumis(name):
    n = name.lower()
    if 'scabies' in n:
        return 'scabies'
    if 'psoria' in n and 'parapsoria' not in n:
        return 'psoriasis'
    if 'tinea' in n or 'dermatophyt' in n or 'ringworm' in n:
        return 'tinea'
    if n.startswith('acne') and 'rosacea' not in n and 'keloid' not in n:
        return 'acne'
    if ('eczema' in n or 'atopic' in n) and 'contact' not in n:
        return 'eczema'
    return None

def main_label(w):
    if not w:
        return None, 0.0
    name, weight = max(w.items(), key=lambda kv: kv[1])
    return to_lumis(name), weight

def fitz(r):
    vals = [r.get(f'dermatologist_fitzpatrick_skin_type_label_{i}') for i in (1, 2, 3)]
    vals = [v for v in vals if isinstance(v, str) and v.startswith('FST')]
    if not vals:
        return None
    v = int(round(np.median([int(x[3]) for x in vals])))
    return 'I-II' if v <= 2 else ('III-IV' if v <= 4 else 'V-VI')

scin['weights'] = scin.weighted_skin_condition_label.map(parse)
scin[['lumis_class', 'agreement']] = scin.weights.apply(lambda w: pd.Series(main_label(w)))
scin['tone'] = scin.apply(fitz, axis=1)

EXTRA_CSV, EXTRA_ZIP = f'{BASE}/data/scin_extra_v1.csv', f'{BASE}/data/scin_extra_images.zip'
known_cases, known_md5 = set(split.case_id), set(split.md5)
if os.path.exists(EXTRA_CSV):
    extra = pd.read_csv(EXTRA_CSV, dtype={'case_id': str})
    if not os.path.exists('/content/scin_extra'):
        with zipfile.ZipFile(EXTRA_ZIP) as f:
            f.extractall('/content')
    print('Loaded saved extra cases')
else:
    pick = scin[scin.lumis_class.notna() & (scin.agreement >= MIN_EXTRA_AGREEMENT) & (scin.agreement < 0.5)
                & ~scin.case_id.isin(known_cases)]
    os.makedirs('/content/scin_extra', exist_ok=True)
    rows = []
    for _, r in pick.iterrows():
        for i in (1, 2, 3):
            p = r.get(f'image_{i}_path')
            if not isinstance(p, str):
                continue
            name = f'scin_extra/{r.case_id}_{i}{os.path.splitext(p)[1] or ".png"}'
            try:
                data = fetch(p)
            except Exception as e:
                print('skipped', p, type(e).__name__)
                continue
            open(f'/content/{name}', 'wb').write(data)
            rows.append({'path': name, 'case_id': r.case_id, 'image_no': i, 'lumis_class': r.lumis_class,
                         'agreement': round(float(r.agreement), 3), 'tone': r.tone, 'md5': hashlib.md5(data).hexdigest()})
    extra = pd.DataFrame(rows, columns=['path', 'case_id', 'image_no', 'lumis_class', 'agreement', 'tone', 'md5'])
    extra = extra.drop_duplicates(['case_id', 'md5'])
    clash = set(extra[extra.md5.isin(known_md5)].case_id)          # same photo as a split case: drop the whole case
    extra = extra[~extra.case_id.isin(clash)].reset_index(drop=True)
    print(f'Dropped {len(clash)} extra cases that share a photo with the frozen split')
    with zipfile.ZipFile('/content/scin_extra_images.zip', 'w', zipfile.ZIP_STORED) as z:
        for p in extra.path:
            z.write(f'/content/{p}', p)
    import shutil
    shutil.copy('/content/scin_extra_images.zip', EXTRA_ZIP)
    extra.to_csv(EXTRA_CSV, index=False)
extra['file'] = '/content/' + extra.path
print(f'Extra: {len(extra)} photos from {extra.case_id.nunique()} cases')
print(extra.drop_duplicates('case_id').lumis_class.value_counts().to_dict())

## 2. Folds (rule 1)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

parent = {c: c for c in tv.case_id.unique()}
def find(c):
    while parent[c] != c:
        parent[c] = parent[parent[c]]
        c = parent[c]
    return c
for _, g in tv.groupby('md5'):
    cs = g.case_id.unique()
    for c in cs[1:]:
        parent[find(c)] = find(cs[0])
tv['group'] = tv.case_id.map(find)

ev = tv[tv.eval_photo].reset_index(drop=True)                    # one close-up per case: what gets scored
y_ev = ev.lumis_class.map(IDX).values
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_of = {}
for f, (_, te) in enumerate(sgkf.split(ev, y_ev, ev.group)):
    for c in ev.case_id.iloc[te]:
        fold_of[c] = f
tv['fold'] = tv.case_id.map(fold_of)
ev['fold'] = ev.case_id.map(fold_of)
print(pd.crosstab(ev.lumis_class, ev.fold, margins=True))

## 3. Embeddings (saved to Drive)

In [ ]:
# EMBEDDING FUNCTIONS
import tensorflow as tf
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)
from PIL import Image, ImageOps

def open_rgb(path):
    return ImageOps.exif_transpose(Image.open(path)).convert('RGB')

_loaded = {}

def embed_derm_foundation(paths, batch=8):
    # Google exported this model for CPU only ("CUDA is not among the platforms required by the module: [CPU]"),
    # so it runs on the CPU even on a GPU runtime. Slow, but the results are saved in chunks and resume.
    from huggingface_hub import snapshot_download
    with tf.device('/CPU:0'):
        if 'df' not in _loaded:
            _loaded['df'] = tf.saved_model.load(snapshot_download('google/derm-foundation', token=HF_TOKEN))
        infer = _loaded['df'].signatures['serving_default']
        return _derm_foundation_batches(infer, paths, batch)

def _derm_foundation_batches(infer, paths, batch):
    out = []
    for i in range(0, len(paths), batch):
        examples = []
        for p in paths[i:i + batch]:
            buf = io.BytesIO()
            open_rgb(p).resize((448, 448), Image.BILINEAR).save(buf, 'PNG')
            examples.append(tf.train.Example(features=tf.train.Features(feature={'image/encoded': tf.train.Feature(
                bytes_list=tf.train.BytesList(value=[buf.getvalue()]))})).SerializeToString())
        out.append(infer(inputs=tf.constant(examples))['embedding'].numpy().reshape(len(examples), -1))
    return np.concatenate(out).astype(np.float32)

def embed_medsiglip(paths, batch=32):
    import torch
    from transformers import AutoModel, AutoProcessor
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    if 'ms' not in _loaded:
        _loaded['ms'] = (AutoModel.from_pretrained('google/medsiglip-448', token=HF_TOKEN).to(dev).eval(),
                         AutoProcessor.from_pretrained('google/medsiglip-448', token=HF_TOKEN))
    model, proc = _loaded['ms']
    out = []
    with torch.no_grad():
        for i in range(0, len(paths), batch):
            x = proc(images=[open_rgb(p) for p in paths[i:i + batch]], return_tensors='pt').to(dev)
            f = model.get_image_features(pixel_values=x['pixel_values'])
            f = getattr(f, 'pooler_output', f)
            out.append(f.float().cpu().numpy())
    return np.concatenate(out).astype(np.float32)

EMBED = {'derm_foundation': embed_derm_foundation, 'medsiglip': embed_medsiglip}

In [ ]:
def readable(files):
    ok = []
    for f in files:
        try:
            Image.open(f).verify()
            ok.append(f)
        except Exception:
            print('cannot be read, left out:', f)
    return set(ok)

good = readable(list(d_train.file) + list(d_val.file) + list(tv.file) + list(extra.file))
d_train, d_val = d_train[d_train.file.isin(good)].reset_index(drop=True), d_val[d_val.file.isin(good)].reset_index(drop=True)
extra = extra[extra.file.isin(good)].reset_index(drop=True)
print('SCIN train/val photos left out (unreadable):', int((~tv.file.isin(good)).sum()))
tv = tv[tv.file.isin(good)].reset_index(drop=True)

ALL = list(dict.fromkeys(list(d_train.file) + list(d_val.file) + list(tv.file) + list(extra.file)))
EMB_DIR = f'{BASE}/exports/embeddings'
os.makedirs(EMB_DIR, exist_ok=True)
CHUNK = 256
KEY = hashlib.md5('\n'.join(ALL).encode()).hexdigest()[:10]      # changes if the photo list changes

def cached_embed(bb):
    d = f'{EMB_DIR}/{bb}'
    os.makedirs(d, exist_ok=True)
    n, parts, t0, new = (len(ALL) + CHUNK - 1) // CHUNK, [], time.time(), 0
    for k in range(n):
        p = f'{d}/{KEY}_{k:04d}.npy'
        if not os.path.exists(p):
            np.save(p, EMBED[bb](ALL[k * CHUNK:(k + 1) * CHUNK]))
            new += 1
            per = (time.time() - t0) / (new * CHUNK)
            print(f'{bb}: chunk {k + 1}/{n} | {per:.2f} s per photo | about {per * CHUNK * (n - k - 1) / 60:.0f} min left')
        parts.append(np.load(p))
    return np.concatenate(parts)

emb, row = {}, {f: i for i, f in enumerate(ALL)}
for bb in sorted(BACKBONES, key=lambda b: b == 'derm_foundation'):   # the fast GPU model first
    emb[bb] = cached_embed(bb)
    print(bb, emb[bb].shape, '(saved in Drive; a rerun after a disconnect resumes)')

## 4. v1 on the same cases (baseline)

In [ ]:
import keras
v1 = keras.models.load_model(v1info.model_file)

def v1_logits(files):
    def load(path):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img = tf.cast(tf.cast(tf.image.resize(img, (224, 224)), tf.uint8), tf.float32)
        return keras.applications.efficientnet.preprocess_input(img)
    ds = tf.data.Dataset.from_tensor_slices(list(files)).map(load).batch(32)
    return v1.predict(ds, verbose=0) + ADJ

ev = tv[tv.eval_photo].reset_index(drop=True)
y_ev = ev.lumis_class.map(IDX).values
z_v1 = v1_logits(ev.file)
z_v1_dval = v1_logits(d_val.file)
y_dval = d_val.lumis_class.map(IDX).values
print('v1 on SCIN (482 cases):', round(float((z_v1.argmax(1) == y_ev).mean()), 4),
      '| v1 on DermNet val:', round(float((z_v1_dval.argmax(1) == y_dval).mean()), 4))

## 5. Cross-validated candidates (rules 1, 2 and 4)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score

def balanced(labels, cap=None):
    labels = np.asarray(labels)
    w = len(labels) / (K * np.maximum(np.bincount(labels, minlength=K), 1))
    return (np.minimum(w, cap) if cap else w)[labels]

def macro_sens(y, p):
    return float(np.mean([(p[y == k] == k).mean() for k in range(K) if (y == k).any()]))

def scores(y, p):
    s = {'accuracy': float((p == y).mean()), 'macro_f1': float(f1_score(y, p, average='macro', labels=range(K), zero_division=0)),
         'macro_sensitivity': macro_sens(y, p)}
    for k, c in enumerate(CLASSES):
        s[f'sens_{c}'] = float((p[y == k] == k).mean()) if (y == k).any() else np.nan
    return s

yd = d_train.lumis_class.map(IDX).values
wd = balanced(yd)

def training_set(bb, scin_rows, use_extra):
    pool = pd.concat([scin_rows.assign(mult=1.0), extra.assign(mult=extra.agreement)] if use_extra else [scin_rows.assign(mult=1.0)])
    yp = pool.lumis_class.map(IDX).values
    X = np.concatenate([emb[bb][[row[f] for f in d_train.file]], emb[bb][[row[f] for f in pool.file]]])
    y = np.concatenate([yd, yp])
    w = np.concatenate([wd, SCIN_W * balanced(yp, MAX_WEIGHT) * pool.mult.values])
    return X, y, w

def fit(X, y, w, C, prep=None):
    if prep is None:
        sc = StandardScaler().fit(X)
        pca = PCA(PCA_DIM, random_state=SEED).fit(sc.transform(X))
        prep = (sc, pca)
    clf = LogisticRegression(C=C, max_iter=3000).fit(prep[1].transform(prep[0].transform(X)), y, sample_weight=w)
    return clf, prep

def logits(clf, prep, X):
    return clf.decision_function(prep[1].transform(prep[0].transform(X))) + ADJ

results, oofs, chosen = {}, {}, {}
for bb in BACKBONES:
    for use_extra in (False, True):
        name = f'{bb}{" + extra" if use_extra else ""}'
        t0 = time.time()
        oof, cs = np.zeros((len(ev), K)), []
        for f in range(N_FOLDS):
            tr = tv[tv.fold != f]
            X, y, w = training_set(bb, tr, use_extra)
            _, prep = fit(X, y, w, 1.0)                                # PCA fitted on this training fold
            tr_ev = tr[tr.eval_photo].reset_index(drop=True)
            inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
            c_score = {C: [] for C in C_GRID}
            for _, iv in inner.split(tr_ev, tr_ev.lumis_class, tr_ev.group):
                held = set(tr_ev.case_id.iloc[iv])
                Xi, yi, wi = training_set(bb, tr[~tr.case_id.isin(held)], use_extra)
                Xv = emb[bb][[row[p] for p in tr_ev.file.iloc[iv]]]
                yv = tr_ev.lumis_class.iloc[iv].map(IDX).values
                for C in C_GRID:
                    clf, _ = fit(Xi, yi, wi, C, prep)
                    c_score[C].append(macro_sens(yv, logits(clf, prep, Xv).argmax(1)))
            best_C = max(C_GRID, key=lambda C: (np.mean(c_score[C]), -C))
            cs.append(best_C)
            clf, _ = fit(X, y, w, best_C, prep)
            te = np.where(ev.fold.values == f)[0]
            oof[te] = logits(clf, prep, emb[bb][[row[p] for p in ev.file.iloc[te]]])
        X, y, w = training_set(bb, tv, use_extra)                    # DermNet check: fit on all train+val
        clf, prep = fit(X, y, w, max(set(cs), key=cs.count))
        dval_acc = float((logits(clf, prep, emb[bb][[row[p] for p in d_val.file]]).argmax(1) == y_dval).mean())
        results[name] = {**scores(y_ev, oof.argmax(1)), 'dermnet_val_accuracy': dval_acc}
        oofs[name], chosen[name] = oof, cs
        print(f'{name}: C per fold {cs} | {time.time() - t0:.0f} s')

results['v1'] = {**scores(y_ev, z_v1.argmax(1)), 'dermnet_val_accuracy': float((z_v1_dval.argmax(1) == y_dval).mean())}

## 6. Comparison: Claude asked directly (no Lumis model)

Claude Haiku 5.5 sees the same close-up photo and must pick one of the five conditions. Thinking is left on (its
default), so it can reason as a general AI normally would. A failed call counts as a wrong answer.

In [ ]:
import anthropic, base64
NAMES = {'acne': 'Acne', 'eczema': 'Eczema (atopic dermatitis)', 'psoriasis': 'Psoriasis', 'scabies': 'Scabies',
         'tinea': 'Fungal skin infection (tinea or ringworm)'}
BY_NAME = {v: k for k, v in NAMES.items()}
ZS_SCHEMA = {'type': 'object', 'properties': {'condition': {'type': 'string', 'enum': list(NAMES.values())}},
             'required': ['condition'], 'additionalProperties': False}
ZS_PROMPT = ('Someone took this photo of a skin problem with their phone. Which one of these five conditions is it most '
             'consistent with? ' + '; '.join(NAMES.values()) + '. Choose the single most likely one.')
client = anthropic.Anthropic(api_key=ANTHROPIC_KEY, timeout=60, max_retries=2)

def jpeg_b64(path, max_edge=1024):
    im = open_rgb(path)
    im.thumbnail((max_edge, max_edge), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, 'JPEG', quality=90)
    return base64.standard_b64encode(buf.getvalue()).decode()

def ask(model, path):
    try:
        r = client.messages.create(model=model, max_tokens=4096, system='Answer only with the requested JSON.',
            messages=[{'role': 'user', 'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': jpeg_b64(path)}},
                {'type': 'text', 'text': ZS_PROMPT}]}],
            extra_body={'output_config': {'format': {'type': 'json_schema', 'schema': ZS_SCHEMA}}})
        text = next((b.text for b in r.content if getattr(b, 'type', None) == 'text'), None)
        return {'ok': True, 'condition': BY_NAME[json.loads(text)['condition']]}
    except Exception as e:
        return {'ok': False, 'error': type(e).__name__}

zs_dir = f'{BASE}/exports/zero_shot'
os.makedirs(zs_dir, exist_ok=True)
for model in ZERO_SHOT_MODELS:
    cache_file = f'{zs_dir}/answers_{model}.jsonl'
    cache, failed = {}, set()
    if os.path.exists(cache_file):
        for line in open(cache_file):
            r = json.loads(line)
            (cache.__setitem__(r['file'], r) if r['ok'] else failed.add(r['file']))
    t0 = time.time()
    with open(cache_file, 'a') as fh:
        for f in ev.file:
            if f not in cache:
                r = {'file': f, **ask(model, f)}
                fh.write(json.dumps(r) + '\n')
                if r['ok']:
                    cache[f] = r
                else:
                    failed.add(f)
    pred = np.array([IDX[cache[f]['condition']] if f in cache else -1 for f in ev.file])
    results[f'Claude ({model}), asked directly'] = {**scores(y_ev, pred), 'dermnet_val_accuracy': np.nan,
                                                     'first_try_failed': len(failed & set(ev.file))}
    print(f'{model}: {len(ev)} photos in {time.time() - t0:.0f} s, calls that ever failed: {len(failed & set(ev.file))}')

## 7. Results and decision (rules 2 and 3)

In [ ]:
table = pd.DataFrame(results).T
cols = ['accuracy', 'macro_sensitivity', 'macro_f1'] + [f'sens_{c}' for c in CLASSES] + ['dermnet_val_accuracy']
display(table[cols].round(3))

rng = np.random.default_rng(SEED)
boots = [rng.integers(0, len(ev), len(ev)) for _ in range(2000)]
def diff_ci(name):
    p1, p0 = oofs[name].argmax(1), z_v1.argmax(1)
    d = [macro_sens(y_ev[b], p1[b]) - macro_sens(y_ev[b], p0[b]) for b in boots]
    return [round(float(np.percentile(d, 2.5)), 3), round(float(np.percentile(d, 97.5)), 3)]

base = results['v1']
checks, passing = {}, []
for name in oofs:
    r = results[name]
    c = {'average sensitivity higher': r['macro_sensitivity'] > base['macro_sensitivity'],
         'macro F1 higher': r['macro_f1'] > base['macro_f1'],
         'tinea sensitivity not lower': r['sens_tinea'] >= base['sens_tinea'],
         'accuracy not lower': r['accuracy'] >= base['accuracy'],
         'DermNet drop at most 3 points': r['dermnet_val_accuracy'] >= base['dermnet_val_accuracy'] - MAX_DERMNET_DROP}
    checks[name] = {**c, 'macro_sensitivity_gain_ci95': diff_ci(name)}
    print(f'\n{name}  (gain in average sensitivity {r["macro_sensitivity"] - base["macro_sensitivity"]:+.3f}, '
          f'95% CI {checks[name]["macro_sensitivity_gain_ci95"]})')
    for k, ok in c.items():
        print(f'   {"ok  " if ok else "FAIL"}  {k}')
    if all(c.values()):
        passing.append(name)

winner = None
if passing:
    winner = max(passing, key=lambda n: (round(results[n]['macro_sensitivity'], 6), '+ extra' not in n))
print('\nWINNER:', winner if winner else 'none - v1 stays (rule 3)')

print('\nAccuracy by skin tone (SCIN, scored cases):')
tone = ev.tone.fillna('unknown').values
by_tone = {n: {t: round(float((p[tone == t] == y_ev[tone == t]).mean()), 3) for t in sorted(set(tone))}
           for n, p in [('v1', z_v1.argmax(1))] + ([(winner, oofs[winner].argmax(1))] if winner else [])}
by_tone['cases'] = {t: int((tone == t).sum()) for t in sorted(set(tone))}
display(pd.DataFrame(by_tone))

In [ ]:
table.to_csv(f'{OUT}/cv_results.csv')
np.savez(f'{OUT}/oof_logits.npz', y=y_ev, v1=z_v1, **{n.replace(' + ', '_plus_'): o for n, o in oofs.items()})
ev[['path', 'case_id', 'lumis_class', 'tone', 'fold']].to_csv(f'{OUT}/scored_cases.csv', index=False)
json.dump({'winner': winner, 'v1_run': V1_RUN, 'chosen_C': chosen, 'checks': checks,
           'results': {k: {m: (None if pd.isna(v) else v) for m, v in r.items()} for k, r in results.items()},
           'settings': {'n_folds': N_FOLDS, 'pca_dim': PCA_DIM, 'c_grid': C_GRID, 'scin_w': SCIN_W, 'max_weight': MAX_WEIGHT,
                        'min_extra_agreement': MIN_EXTRA_AGREEMENT, 'zero_shot_models': ZERO_SHOT_MODELS},
           'created_at': time.strftime('%Y-%m-%d %H:%M')},
          open(f'{OUT}/decision.json', 'w'), indent=1, default=float)
print('Saved', f'{OUT}/decision.json')